# Part 2: ResNet-18 Model Improvement Ablations

This notebook keeps the Part 2 workflow notebook-owned while moving reusable training and result logic into project modules. The regular-training baseline is loaded from Part 1 results; the remaining ablations are trained here.

In [1]:
from pathlib import Path
import importlib
import os
import sys

In [2]:
current = Path.cwd().resolve()
for candidate in [current, *current.parents]:
    if (candidate / 'src').is_dir() and (candidate / 'requirements.txt').exists():
        ROOT = candidate
        break
else:
    ROOT = current

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

In [3]:
def install_project_requirements_for_colab(project_root: Path) -> None:
    """Install non-PyTorch dependencies in Colab without replacing CUDA-matched torch wheels."""
    try:
        import google.colab  # type: ignore  # noqa: F401
    except ImportError:
        return

    import subprocess

    requirements_path = project_root / 'requirements.txt'
    filtered_requirements = Path('/tmp/mlds_colab_requirements.txt')
    skip_prefixes = ('torch', 'torchvision')
    filtered_lines = [
        line
        for line in requirements_path.read_text().splitlines()
        if not line.strip().lower().startswith(skip_prefixes)
    ]
    filtered_requirements.write_text('\n'.join(filtered_lines) + '\n')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(filtered_requirements)])


In [4]:
install_project_requirements_for_colab(ROOT)

ROOT

PosixPath('/Users/royrubin/Documents/GitHub/MLDS_Final_Project')

In [5]:
import pandas as pd
from IPython.core.display import Image
from IPython.display import display

import src.evaluation.experiment_results as experiment_results

experiment_results = importlib.reload(experiment_results)
import src.experiments.part2 as part2_experiments  # noqa: E402
part2_experiments = importlib.reload(part2_experiments)
experiment_output_paths = experiment_results.experiment_output_paths
get_device = experiment_results.get_device
load_part1_model_baseline_aggregated = experiment_results.load_part1_model_baseline_aggregated
run_part2_improvement_experiments = part2_experiments.run_part2_improvement_experiments
from src.utils.config import Part2ExperimentConfig  # noqa: E402
from src.utils.reproducibility import seed_everything  # noqa: E402

/Users/royrubin/opt/anaconda3/envs/mlds_dogs_vs_cats/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Configuration

In [6]:
config = Part2ExperimentConfig()
device = get_device(config)
output_paths = experiment_output_paths(config.results_dir, config.figures_dir, config.part)
seed_everything(config.seed, deterministic=config.deterministic)

pd.DataFrame(
    [
        {
            'part': config.part,
            'config_name': config.config_name,
            'model_name': config.model_name,
            'tiles_per_side_values': config.tiles_per_side_values,
            'num_tile_permutations': config.num_tile_permutations,
            'epochs': config.epochs,
            'device': str(device),
        }
    ]
)

Selected device: cpu


,part,config_name,model_name,tiles_per_side_values,num_tile_permutations,epochs,device
0,part2,part2_improvement,resnet18,"[1, 3]",2,3,cpu


In [7]:
pd.DataFrame(config.ablations)

,name,use_pretrained,use_standard_augmentation,freeze_backbone
0,augmentation_only,True,True,False
1,finetune_only,True,False,False
2,augmentation_finetune,True,True,False


## Part 1 ResNet-18 Baseline

In [8]:
part1_baseline_aggregated = load_part1_model_baseline_aggregated(config, config.model_name)
if not part1_baseline_aggregated.empty:
    display(part1_baseline_aggregated)

,model_name,tiles_per_side,num_tiles,mean_final_epoch_val_accuracy,std_final_epoch_val_accuracy,mean_best_epoch_val_accuracy,std_best_epoch_val_accuracy,n_runs,ablation_name,config_name
0,resnet18,3.0,9,0.894231,0.013598,0.894231,0.013598,2,regular_part1,part2_improvement
1,resnet18,NaN,1,0.865385,NaN,0.903846,NaN,1,regular_part1,part2_improvement


## Train Improvement Ablations

In [9]:
RUN_TRAINING = True

if RUN_TRAINING:
    part2_results = run_part2_improvement_experiments(config=config, device=device)
    display(part2_results)
else:
    print('Training is skipped. Set RUN_TRAINING = True to train Part 2 ablations in this notebook.')

Saved 9 pending row(s) for model 'resnet18' to /Users/royrubin/Documents/GitHub/MLDS_Final_Project/outputs/results/part2_raw_results.csv.

Running ablation: augmentation_only

[1/3] model=resnet18, ablation=augmentation_only, tiles_per_side=None, tile_permutation_id=0, seed=42
Building dataloaders...
Built dataloaders: 13 train batches, 4 validation batches.
Building model 'resnet18'...
Best checkpoint path: /Users/royrubin/Documents/GitHub/MLDS_Final_Project/outputs/checkpoints/part2/part2_20260512_035614/resnet18__augmentation_only__tiles_1__perm_0__best.pt
Raw results path: /Users/royrubin/Documents/GitHub/MLDS_Final_Project/outputs/results/part2_raw_results.csv


resnet18 augmentation_only 1x1 permutation 0: 100%|██████████| 3/3 [02:28<00:00, 49.47s/epoch, best_val_accuracy=0.981, train_accuracy=0.971, train_loss=0.0921, val_accuracy=0.981, val_loss=0.0662]



[2/3] model=resnet18, ablation=augmentation_only, tiles_per_side=3, tile_permutation_id=1, seed=42
Building dataloaders...
Built dataloaders: 13 train batches, 4 validation batches.
Building model 'resnet18'...
Best checkpoint path: /Users/royrubin/Documents/GitHub/MLDS_Final_Project/outputs/checkpoints/part2/part2_20260512_035614/resnet18__augmentation_only__tiles_3__perm_1__best.pt
Raw results path: /Users/royrubin/Documents/GitHub/MLDS_Final_Project/outputs/results/part2_raw_results.csv


resnet18 augmentation_only 3x3 permutation 1: 100%|██████████| 3/3 [02:59<00:00, 59.88s/epoch, best_val_accuracy=0.904, train_accuracy=0.892, train_loss=0.243, val_accuracy=0.865, val_loss=0.346]



[3/3] model=resnet18, ablation=augmentation_only, tiles_per_side=3, tile_permutation_id=2, seed=42
Building dataloaders...
Built dataloaders: 13 train batches, 4 validation batches.
Building model 'resnet18'...
Best checkpoint path: /Users/royrubin/Documents/GitHub/MLDS_Final_Project/outputs/checkpoints/part2/part2_20260512_035614/resnet18__augmentation_only__tiles_3__perm_2__best.pt
Raw results path: /Users/royrubin/Documents/GitHub/MLDS_Final_Project/outputs/results/part2_raw_results.csv


resnet18 augmentation_only 3x3 permutation 2:   0%|          | 0/3 [00:58<?, ?epoch/s]


KeyboardInterrupt: 

## Results

In [ ]:
results_path = Path(output_paths['aggregated_results'])

if results_path.exists():
    part2_results = pd.read_csv(results_path)
    if 'regular_part1' not in set(part2_results.get('ablation_name', [])) and not part1_baseline_aggregated.empty:
        part2_results = pd.concat([part1_baseline_aggregated, part2_results], ignore_index=True, sort=False)
    display(part2_results.sort_values(['tiles_per_side', 'ablation_name']))
else:
    print('Part 2 aggregated results were not found. Set RUN_TRAINING = True and run the training cell.')
    if part1_baseline_aggregated.empty:
        print('Part 1 ResNet-18 baseline results were also not found; run Part 1 first to include regular_part1.')

In [ ]:
figure_path = Path(output_paths['figure'])
if figure_path.exists():
    display(Image(filename=str(figure_path)))
else:
    print('Part 2 ablation figure has not been generated yet.')